# 02 — Silver: Hospital Dimension

**Pipeline:** Bronze → Silver  
**Source:** WA Government hospital GeoJSON (data.wa.gov.au)  
**Target:** `silver.dim_hospitals` (Delta table)  

Steps:
1. Read GeoJSON from bronze layer
2. Flatten nested GeoJSON structure
3. Extract coordinates, facility type, health service
4. Write as Delta dimension table

In [ ]:
# ------------------------------------------------------------------
# 1. Read WA hospital GeoJSON from bronze layer
#    Uploaded manually to: Files/bronze/wa_gov/hospitals/raw.geojson
# ------------------------------------------------------------------
from pyspark.sql.functions import col, explode, get_json_object

raw = spark.read.option("multiline", "true").json(
    "Files/bronze/wa_gov/hospitals/raw.geojson"
)
print("Raw GeoJSON schema:")
raw.printSchema()

In [ ]:
# ------------------------------------------------------------------
# 2. Flatten GeoJSON: explode features array
# ------------------------------------------------------------------
features = raw.select(explode(col("features")).alias("feature"))

print("Feature schema:")
features.select("feature").printSchema()
features.show(3, truncate=False)

In [ ]:
# ------------------------------------------------------------------
# 3. Extract fields from nested properties and geometry
# ------------------------------------------------------------------
dim_hospitals = features.select(
    col("feature.properties.FACILITY_NAME").alias("hospital_name"),
    col("feature.properties.FACILITY_TYPE").alias("facility_type"),
    col("feature.properties.HEALTH_SERVICE").alias("health_service"),
    col("feature.properties.SITE_ID").alias("site_id"),
    col("feature.properties.ADDRESS").alias("address"),
    col("feature.properties.SUBURB").alias("suburb"),
    col("feature.properties.POSTCODE").alias("postcode"),
    col("feature.geometry.coordinates")[0].alias("longitude"),
    col("feature.geometry.coordinates")[1].alias("latitude")
)

print(f"Hospital records: {dim_hospitals.count()}")
dim_hospitals.show(10, truncate=False)

In [ ]:
# ------------------------------------------------------------------
# 4. Quick profile — facility types and health services
# ------------------------------------------------------------------
print("Facility types:")
dim_hospitals.groupBy("facility_type").count().orderBy("count", ascending=False).show()

print("Health services:")
dim_hospitals.groupBy("health_service").count().orderBy("count", ascending=False).show()

In [ ]:
# ------------------------------------------------------------------
# 5. Write silver Delta table
# ------------------------------------------------------------------
dim_hospitals.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.dim_hospitals")

print("silver.dim_hospitals written successfully")
print(f"Total hospitals: {spark.table('silver.dim_hospitals').count()}")